# K-Nearest Neighbor (KNN) - Day 2 (From Scratch)

## Statistical Methods for Selecting k

### 1. Cross-Validation

Cross-Validation is a good way to find the best value of k. This means dividing the dataset into k parts. The model is trained on some of these parts and tested on the remaining ones. This process is repeated for each part.

### 2. Elbow Method

In Elbow Method we draw a graph showing the error rate or accuracy for different k values. As k increases the error usually drops at first. But after a certain point error stops decreasing quickly. The point where the curve changes direction and looks like an "elbow" is usually the best choice for k.

### 3. Odd Values for k

It's a good idea to use an odd number for k, especially in classification problems. This helps avoid ties when deciding which class is the most common among the neighbors.

---

## Distance Metrics Used in KNN Algorithm

### 1. Euclidean Distance

$$d(x, X_i) = \sqrt{\sum_{j=1}^{n} (x_j - X_{ij})^2}$$

### 2. Manhattan Distance

$$d(x, y) = \sum_{i=1}^{n} |x_i - y_i|$$

### 3. Minkowski Distance

$$d(x, y) = \left( \sum_{i=1}^{n} |x_i - y_i|^p \right)^{\frac{1}{p}}$$

- When p=2 → Euclidean distance
- When p=1 → Manhattan distance

---

## One-Line Summary

**Select optimal k using cross-validation and elbow method, with distance metrics like Euclidean, Manhattan, or Minkowski.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("="*50)
print("KNN DAY 2 - SELECTING K AND DISTANCE METRICS")
print("="*50)

## Distance Metrics from Scratch

In [ ]:
def euclidean_distance(x1, x2):
    """Calculate Euclidean distance"""
    return np.sqrt(np.sum((x1 - x2) ** 2))

def manhattan_distance(x1, x2):
    """Calculate Manhattan distance"""
    return np.sum(np.abs(x1 - x2))

def minkowski_distance(x1, x2, p):
    """Calculate Minkowski distance"""
    return np.sum(np.abs(x1 - x2) ** p) ** (1/p)

# Test distances
x1 = np.array([1, 2, 3])
x2 = np.array([4, 5, 6])

print("Distance Metrics Test:")
print(f"x1: {x1}")
print(f"x2: {x2}")
print("="*40)
print(f"Euclidean: {euclidean_distance(x1, x2):.4f}")
print(f"Manhattan: {manhattan_distance(x1, x2):.4f}")
print(f"Minkowski (p=3): {minkowski_distance(x1, x2, p=3):.4f}")

## KNN with Different Distance Metrics

In [ ]:
class KNNWithMetrics:
    """KNN with multiple distance metrics"""
    
    def __init__(self, k=3, metric='euclidean', p=2):
        self.k = k
        self.metric = metric
        self.p = p
        self.X_train = None
        self.y_train = None
    
    def _distance(self, x1, x2):
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((x1 - x2) ** 2))
        elif self.metric == 'manhattan':
            return np.sum(np.abs(x1 - x2))
        elif self.metric == 'minkowski':
            return np.sum(np.abs(x1 - x2) ** self.p) ** (1/self.p)
        else:
            raise ValueError(f"Unknown metric: {self.metric}")
    
    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
    
    def predict(self, X):
        X = np.array(X)
        predictions = []
        for x in X:
            distances = [self._distance(x, xt) for xt in self.X_train]
            k_indices = np.argsort(distances)[:self.k]
            k_labels = self.y_train[k_indices]
            predictions.append(Counter(k_labels).most_common(1)[0][0])
        return np.array(predictions)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == np.array(y))

In [ ]:
# Create dataset
np.random.seed(42)
X = np.random.randn(100, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

print(f"Dataset shape: {X.shape}")
print(f"Class 0: {np.sum(y == 0)}, Class 1: {np.sum(y == 1)}")

In [ ]:
# Compare distance metrics
metrics = ['euclidean', 'manhattan', 'minkowski']

print("\n" + "="*50)
print("Distance Metrics Comparison")
print("="*50)

for metric in metrics:
    knn = KNNWithMetrics(k=5, metric=metric, p=3)
    knn.fit(X, y)
    acc = knn.score(X, y)
    print(f"Metric = {metric}: Accuracy = {acc*100:.2f}%")

## Cross-Validation and Elbow Method

In [ ]:
def cross_validate_knn(X, y, k_values, k_folds=5):
    """
    Perform k-fold cross-validation for different k values
    """
    n_samples = len(X)
    fold_size = n_samples // k_folds
    
    results = {}
    
    for k in k_values:
        fold_accuracies = []
        
        for fold in range(k_folds):
            # Split data
            start = fold * fold_size
            end = start + fold_size if fold < k_folds - 1 else n_samples
            
            # Test indices
            test_indices = list(range(start, end))
            train_indices = [i for i in range(n_samples) if i not in test_indices]
            
            # Train and test
            X_train = X[train_indices]
            y_train = y[train_indices]
            X_test = X[test_indices]
            y_test = y[test_indices]
            
            # Train KNN
            knn = KNNWithMetrics(k=k)
            knn.fit(X_train, y_train)
            acc = knn.score(X_test, y_test)
            fold_accuracies.append(acc)
        
        results[k] = np.mean(fold_accuracies)
    
    return results

In [ ]:
# Find optimal k
k_values = list(range(1, 21, 2))

print("\n" + "="*50)
print("Cross-Validation Results")
print("="*50)

cv_results = cross_validate_knn(X, y, k_values)

for k, acc in cv_results.items():
    print(f"k = {k:2}: Accuracy = {acc*100:.2f}%")

In [ ]:
# Elbow plot
plt.figure(figsize=(10, 6))
plt.plot(list(cv_results.keys()), list(cv_results.values()), marker='o', linewidth=2)
plt.xlabel('K Value')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Elbow Method - Finding Optimal K')
plt.grid(alpha=0.3)

# Highlight best k
best_k = max(cv_results, key=cv_results.get)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
plt.legend()
plt.show()

print(f"\nBest k value: {best_k} (Accuracy: {cv_results[best_k]*100:.2f}%)")

In [ ]:
# Demonstrate odd k values (tie-breaking)
print("\n" + "="*50)
print("Odd vs Even k Values")
print("="*50)

# Create dataset with potential ties
X_tie = np.array([[0, 0], [1, 0], [0, 1], [1, 1], [2, 2]])
y_tie = np.array([0, 0, 1, 1, 0])

test_point = np.array([[1.1, 1.1]])

print("Training points:")
for i in range(len(X_tie)):
    print(f"  {X_tie[i]} -> {y_tie[i]}")
print(f"\nTest point: {test_point[0]}")

k_even = 2
k_odd = 3

print("\n" + "-"*40)
print(f"With k = {k_even} (even):")
knn_even = KNNWithMetrics(k=k_even)
knn_even.fit(X_tie, y_tie)
pred_even = knn_even.predict(test_point)[0]
print(f"  Prediction: {pred_even}")

print(f"\nWith k = {k_odd} (odd):")
knn_odd = KNNWithMetrics(k=k_odd)
knn_odd.fit(X_tie, y_tie)
pred_odd = knn_odd.predict(test_point)[0]
print(f"  Prediction: {pred_odd}")

print("\nOdd k helps avoid ties in classification!")

In [ ]:
# Day 2 Completed
print("\n" + "="*50)
print("KNN DAY 2 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Cross-Validation for selecting k")
print("- Elbow Method visualization")
print("- Odd values for k (tie-breaking)")
print("- Distance metrics (Euclidean, Manhattan, Minkowski)")
print("- Metric comparison")
print("- k-fold cross-validation implementation")
print("="*50)"
print("\nNext: Day 3 - KNN on Real Dataset (Drug Classifier)")